# Overview Schematic Assets

Generate overview-style schematic assets with a shared visual language:
1. attention matrix icons
2. standalone vertical palette
3. stable / fragile behavior cards


In [ ]:
from pathlib import Path
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, PowerNorm
from matplotlib.path import Path as MplPath
from matplotlib.patches import Circle, FancyArrowPatch, FancyBboxPatch, PathPatch, Polygon

def find_project_root(start: Optional[Path] = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'results').is_dir() and (candidate / 'figs').is_dir():
            return candidate
    raise RuntimeError(f'Could not find project root from {start}')


ROOT = find_project_root()
OUT_DIR = ROOT / 'figs' / 'overview'
OUT_DIR.mkdir(parents=True, exist_ok=True)

N = 9
CMAP = LinearSegmentedColormap.from_list(
    'overview_blue_white_orange',
    ['#356fc3', '#9cc1e8', '#e8f1fb', '#f6f6f1', '#fff1ce', '#ffd676', '#f5a10a'],
    N=256,
).copy()
CMAP.set_bad('#ececec')
MATRIX_NORM = PowerNorm(gamma=0.72, vmin=0.0, vmax=1.0)

TITLE_COLOR = '#222222'
FRAME_GREY = '#d8d8d8'
MASK_GREY = '#ececec'
GRID_WHITE = '#ffffff'
RED = '#cc221f'
GREEN = '#198b44'

FRAGILE_PEAK_CENTER = 0.1
FRAGILE_PEAK_WIDTH = 0.09
FRAGILE_PEAK_HEIGHT = 0.54
FRAGILE_TAIL_BASE = 0.1
FRAGILE_TAIL_HEIGHT = 0.01
FRAGILE_TAIL_DECAY = 1.2

METRIC_TEXT_POS = (-0.22, 0.96)
INTERVENTION_TEXT_POS = (0.52, -0.17)


In [ ]:
def save_and_show(fig, out_stem: str):
    svg_path = OUT_DIR / f'{out_stem}.svg'
    png_path = OUT_DIR / f'{out_stem}.png'
    fig.savefig(svg_path, bbox_inches='tight', facecolor=fig.get_facecolor())
    fig.savefig(png_path, bbox_inches='tight', facecolor=fig.get_facecolor(), dpi=260)
    print('saved ->', svg_path)
    print('saved ->', png_path)
    plt.show()


def make_baseline_matrix(n: int = N) -> np.ndarray:
    mat = np.zeros((n, n), dtype=float)
    for i in range(n):
        js = np.arange(i + 1)
        distances = i - js
        recency = np.exp(-distances / 1.6)
        left_sink = 0.52 * np.exp(-js / 1.55)
        diag_boost = np.zeros_like(recency)
        diag_boost[-1] = 2.6 if i > 0 else 3.0
        local_prev = np.zeros_like(recency)
        if i >= 1:
            local_prev[-2] = 0.38
        if i >= 2:
            local_prev[-3] = 0.18
        weights = 0.52 * recency + left_sink + diag_boost + local_prev
        mat[i, :i + 1] = weights / weights.sum()
    return mat


def suppress_diagonal(mat: np.ndarray, keep_ratio: float = 0.18) -> np.ndarray:
    out = mat.copy()
    for i in range(out.shape[0]):
        out[i, i] = out[i, i] * keep_ratio
    return out


baseline = make_baseline_matrix()
diag_suppressed = suppress_diagonal(baseline)


def draw_matrix(ax, matrix: np.ndarray, title: Optional[str] = None):
    visible = np.tril(matrix)
    masked = np.ma.masked_where(np.triu(np.ones_like(visible), k=1).astype(bool), visible)
    ax.imshow(masked, cmap=CMAP, norm=MATRIX_NORM, interpolation='nearest')
    ax.set_facecolor('white')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticks(np.arange(-0.5, matrix.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, matrix.shape[0], 1), minor=True)
    ax.grid(which='minor', color=GRID_WHITE, linewidth=2.1)
    ax.tick_params(which='minor', bottom=False, left=False)
    for spine in ax.spines.values():
        spine.set_visible(False)
    if title:
        ax.set_title(title, fontsize=19, fontweight='semibold', color=TITLE_COLOR, pad=14, fontfamily='DejaVu Serif')


def plot_schematic(matrix: np.ndarray, out_stem: str, title: Optional[str] = None):
    fig, ax = plt.subplots(figsize=(3.6, 3.6), dpi=220)
    fig.patch.set_facecolor('white')
    draw_matrix(ax, matrix, title=title)
    fig.tight_layout(pad=0.3)
    save_and_show(fig, out_stem)


def plot_attention_pair(baseline: np.ndarray, suppressed: np.ndarray, out_stem: str = 'schematic_attention_pair'):
    fig = plt.figure(figsize=(10.6, 4.15), dpi=220, facecolor='white')
    gs = fig.add_gridspec(1, 3, width_ratios=[1.0, 0.23, 1.0], wspace=0.06)
    ax_left = fig.add_subplot(gs[0, 0])
    ax_mid = fig.add_subplot(gs[0, 1])
    ax_right = fig.add_subplot(gs[0, 2])

    draw_matrix(ax_left, baseline, title='baseline attention')
    draw_matrix(ax_right, suppressed, title='diagonal-suppressed\n(value-based view)')

    ax_mid.set_axis_off()
    ax_mid.annotate(
        '',
        xy=(0.86, 0.5),
        xytext=(0.14, 0.5),
        xycoords='axes fraction',
        textcoords='axes fraction',
        arrowprops=dict(arrowstyle='simple', fc='#8a8a8a', ec='#8a8a8a', mutation_scale=38, alpha=0.95),
    )

    fig.tight_layout(pad=0.45)
    save_and_show(fig, out_stem)


def plot_palette(out_stem: str = 'schematic_palette'):
    grad = np.linspace(0.0, 1.0, 512)[:, None]
    fig, ax = plt.subplots(figsize=(1.05, 5.1), dpi=220)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    ax.imshow(grad, aspect='auto', cmap=CMAP, vmin=0.0, vmax=1.0, interpolation='nearest', origin='lower')
    ax.set_xticks([])
    ax.set_yticks([0, grad.shape[0] - 1])
    ax.set_yticklabels(['0', '1'])
    ax.tick_params(axis='y', labelsize=12, colors='#303030')
    for spine in ax.spines.values():
        spine.set_edgecolor('#b7b7b7')
        spine.set_linewidth(0.8)
    fig.tight_layout(pad=0.25)
    save_and_show(fig, out_stem)


def draw_bulb_icon(ax, color: str = '#202020'):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    lw = 3.0
    body_verts = [
        (0.38, 0.34),
        (0.33, 0.44), (0.25, 0.53), (0.25, 0.64),
        (0.25, 0.81), (0.38, 0.89), (0.50, 0.89),
        (0.62, 0.89), (0.75, 0.81), (0.75, 0.64),
        (0.75, 0.53), (0.67, 0.44), (0.62, 0.34),
        (0.38, 0.34),
    ]
    body_codes = [
        MplPath.MOVETO,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.LINETO,
    ]
    body = PathPatch(MplPath(body_verts, body_codes), fill=False, edgecolor=color, linewidth=lw, capstyle='round', joinstyle='round')
    ax.add_patch(body)
    ax.plot([0.39, 0.61], [0.27, 0.27], color=color, linewidth=lw, solid_capstyle='round')
    ax.plot([0.39, 0.61], [0.20, 0.20], color=color, linewidth=lw, solid_capstyle='round')
    ax.plot([0.45, 0.55], [0.12, 0.12], color=color, linewidth=lw, solid_capstyle='round')
    highlight_verts = [
        (0.44, 0.71),
        (0.39, 0.69), (0.37, 0.63), (0.38, 0.57),
    ]
    highlight_codes = [MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4]
    highlight = PathPatch(MplPath(highlight_verts, highlight_codes), fill=False, edgecolor=color, linewidth=lw, capstyle='round', joinstyle='round')
    ax.add_patch(highlight)
    rays = [
        ((0.50, 0.98), (0.50, 0.91)),
        ((0.17, 0.77), (0.23, 0.71)),
        ((0.83, 0.77), (0.77, 0.71)),
        ((0.09, 0.54), (0.15, 0.54)),
        ((0.91, 0.54), (0.85, 0.54)),
    ]
    for (x0, y0), (x1, y1) in rays:
        ax.plot([x0, x1], [y0, y1], color=color, linewidth=lw, solid_capstyle='round')


def draw_curved_arrow_icon(ax, color: str = '#303030'):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    lw = 2.0
    top_verts = [
        (0.12, 0.80),
        (0.24, 0.70), (0.47, 0.62), (0.72, 0.62),
        (0.72, 0.70),
        (0.88, 0.54),
        (0.72, 0.38),
        (0.72, 0.46),
        (0.44, 0.46),
        (0.27, 0.47), (0.18, 0.57), (0.12, 0.80),
    ]
    top_codes = [
        MplPath.MOVETO,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
        MplPath.LINETO,
        MplPath.LINETO,
        MplPath.LINETO,
        MplPath.LINETO,
        MplPath.CURVE4,
        MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
    ]
    arrow = PathPatch(MplPath(top_verts, top_codes), fill=False, edgecolor=color, linewidth=lw, capstyle='round', joinstyle='miter')
    ax.add_patch(arrow)


def draw_behavior_icon(ax, kind: str, color: str):
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_aspect('equal')
    ax.axis('off')
    if kind == 'idea':
        draw_bulb_icon(ax, color=color)
    elif kind == 'curved-arrow':
        draw_curved_arrow_icon(ax, color=color)
    elif kind == 'stable':
        ax.add_patch(Circle((0.5, 0.5), 0.43, fill=False, linewidth=2.0, edgecolor=color))
        ax.plot([0.27, 0.45, 0.72], [0.48, 0.23, 0.73], color=color, linewidth=2.8, solid_capstyle='round')
    else:
        tri = Polygon([[0.5, 0.92], [0.08, 0.12], [0.92, 0.12]], closed=True, fill=False, linewidth=2.0, edgecolor=color, joinstyle='round')
        ax.add_patch(tri)
        ax.plot([0.5, 0.5], [0.34, 0.68], color=color, linewidth=3.0, solid_capstyle='round')
        ax.add_patch(Circle((0.5, 0.22), 0.03, color=color))


def make_curve(kind: str):
    x = np.linspace(0.0, 1.0, 250)
    if kind == 'stable':
        y = 0.63 + 0.008 * np.sin(8 * np.pi * x) + 0.01 * x
    else:
        y = (
            FRAGILE_TAIL_BASE
            + FRAGILE_PEAK_HEIGHT * np.exp(-((x - FRAGILE_PEAK_CENTER) / FRAGILE_PEAK_WIDTH) ** 2)
            + FRAGILE_TAIL_HEIGHT * np.exp(-FRAGILE_TAIL_DECAY * x)
        )
        y = np.clip(y, 0.10, 0.86)
    return x, y


def plot_behavior_card(kind: str, out_stem: str):
    is_stable = kind == 'stable'
    color = GREEN if is_stable else RED
    label = 'Good / stable behavior' if is_stable else 'Poor / fragile behavior'
    fill = '#f6fbf7' if is_stable else '#fff7f7'

    fig = plt.figure(figsize=(14.8, 2.2), dpi=220, facecolor='white')
    card = fig.add_axes([0, 0, 1, 1])
    card.set_axis_off()
    card.add_patch(
        FancyBboxPatch(
            (0.012, 0.08),
            0.976,
            0.84,
            boxstyle='round,pad=0.008,rounding_size=0.03',
            linewidth=1.6,
            edgecolor=color,
            facecolor=fill,
        )
    )

    icon_ax = fig.add_axes([0.03, 0.22, 0.11, 0.56])
    draw_behavior_icon(icon_ax, kind=kind, color=color)

    curve_ax = fig.add_axes([0.19, 0.24, 0.39, 0.56])
    curve_ax.set_facecolor('none')
    curve_ax.spines['top'].set_visible(False)
    curve_ax.spines['right'].set_visible(False)
    curve_ax.spines['left'].set_linewidth(1.1)
    curve_ax.spines['bottom'].set_linewidth(1.1)
    curve_ax.spines['left'].set_color('#202020')
    curve_ax.spines['bottom'].set_color('#202020')
    curve_ax.set_xlim(0, 1.03)
    curve_ax.set_ylim(0, 1.0)
    curve_ax.set_xticks([])
    curve_ax.set_yticks([])
    x, y = make_curve(kind)
    curve_ax.plot(x, y, color=color, linewidth=2.2)
    curve_ax.hlines(0.74 if is_stable else 0.82, 0.02, 1.0, colors='#b9b9b9', linestyles=(0, (4, 3)), linewidth=1.0)
    curve_ax.annotate('', xy=(1.03, 0.0), xytext=(0.0, 0.0), arrowprops=dict(arrowstyle='-|>', color='#202020', lw=1.1, mutation_scale=10))
    curve_ax.annotate('', xy=(0.0, 1.0), xytext=(0.0, 0.0), arrowprops=dict(arrowstyle='-|>', color='#202020', lw=1.1, mutation_scale=10))
    curve_ax.text(METRIC_TEXT_POS[0], METRIC_TEXT_POS[1], 'metric', ha='left', va='top', fontsize=12, fontweight='semibold', fontfamily='DejaVu Serif')
    curve_ax.text(INTERVENTION_TEXT_POS[0], INTERVENTION_TEXT_POS[1], 'intervention strength', ha='center', va='top', fontsize=12, fontweight='semibold', fontfamily='DejaVu Serif')

    text_ax = fig.add_axes([0.62, 0.16, 0.33, 0.68])
    text_ax.set_axis_off()
    text_ax.text(0.0, 0.5, label, color=color, fontsize=22, fontweight='bold', va='center', ha='left', fontfamily='DejaVu Serif')

    save_and_show(fig, out_stem)


def plot_icon(kind: str, out_stem: str, color: str = '#202020'):
    fig, ax = plt.subplots(figsize=(1.8, 1.8), dpi=260)
    fig.patch.set_facecolor('white')
    ax.set_facecolor('white')
    draw_behavior_icon(ax, kind=kind, color=color)
    fig.tight_layout(pad=0.02)
    save_and_show(fig, out_stem)


In [ ]:
plot_schematic(baseline, 'schematic_baseline_diag')
plot_schematic(diag_suppressed, 'schematic_diagonal_suppressed')
plot_attention_pair(baseline, diag_suppressed)
plot_palette()


In [ ]:
plot_behavior_card('fragile', 'schematic_fragile_behavior')


In [ ]:
plot_behavior_card('stable', 'schematic_stable_behavior')


In [ ]:
plot_icon('idea', 'schematic_bulb_icon')


In [ ]:
plot_icon('curved-arrow', 'schematic_curved_arrow')
